In [1]:
import pandas as pd
df = pd.read_parquet('df_final.parquet')

In [2]:
from sklearn.preprocessing import OrdinalEncoder
ord_enc = OrdinalEncoder()

for header in list(df.columns):
    if (df[header].dtype == object):
        df[header] = ord_enc.fit_transform(df[[header]])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65704 entries, 0 to 65703
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Tournament           65704 non-null  float64       
 1   Date                 65704 non-null  datetime64[ns]
 2   Series               65704 non-null  float64       
 3   Court                65704 non-null  float64       
 4   Surface              65704 non-null  float64       
 5   Round                65704 non-null  float64       
 6   Best of              65704 non-null  int64         
 7   Player_1             65704 non-null  float64       
 8   Player_2             65704 non-null  float64       
 9   Winner               65704 non-null  float64       
 10  Rank_1               65704 non-null  int64         
 11  Rank_2               65704 non-null  int64         
 12  Pts_1                65704 non-null  int64         
 13  Pts_2                65704 non-

In [3]:
cutoff_date = "2022-01-01"
begin_date = "2018-01-01"
train = df[(df["Date"] < cutoff_date) & (df["Date"] >= begin_date)]
test = df[df["Date"] >= cutoff_date]

In [4]:
def make_training_rows(df):
    rows = []
    for _, row in df.iterrows():
        w, l = row["Winner"], row["Loser"]

        # features
        w_feats = {
            "elo_diff": row["winner_elo_pre"] - row["loser_elo_pre"],
            "elo_surf_diff": row["winner_elo_surf_pre"] - row["loser_elo_surf_pre"],
            "h2h_pre": row["h2h_pre"],
            "recent_form_diff": row["recent_form_diff"],
            "label": 1  # winner perspective
        }
        l_feats = {
            "elo_diff": row["loser_elo_pre"] - row["winner_elo_pre"],
            "elo_surf_diff": row["loser_elo_surf_pre"] - row["winner_elo_surf_pre"],
            "h2h_pre": -row["h2h_pre"],  # flip perspective
            "recent_form_diff": -row["recent_form_diff"],
            "label": 0  # loser perspective
        }

        rows.append(w_feats)
        rows.append(l_feats)

    return pd.DataFrame(rows)


In [5]:
train_data = make_training_rows(train)
test_data = make_training_rows(test)

X_train = train_data.drop(columns=["label"])
y_train = train_data["label"]

X_test = test_data.drop(columns=["label"])
y_test = test_data["label"]

print(y_train.value_counts())  # should now show both 0 and 1

label
1    8698
0    8698
Name: count, dtype: int64


In [6]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd

# Assume your data is already loaded and split as in your prompt
# X_train, y_train, X_test, y_test

# Convert pandas DataFrames to numpy arrays
X_train_np = X_train.values.astype(np.float32)
y_train_np = y_train.values.astype(np.int64) # Use int64 for CrossEntropyLoss

X_test_np = X_test.values.astype(np.float32)
y_test_np = y_test.values.astype(np.int64)

# Create a custom PyTorch Dataset
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create Dataset and DataLoader instances
train_dataset = TabularDataset(X_train_np, y_train_np)
test_dataset = TabularDataset(X_test_np, y_test_np)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Define number of input features and classes
input_dim = X_train.shape[1]
output_dim = len(y_train.value_counts())

In [7]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, output_dim):
        super(MLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, hidden_dim2//2),
            nn.ReLU(),
            nn.Linear(hidden_dim2//2, output_dim)
        )

    def forward(self, x):
        return self.layers(x)

# Model, Loss, and Optimizer
model_mlp = MLP(input_dim=input_dim, hidden_dim1=256, hidden_dim2=64, output_dim=output_dim)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_mlp.parameters(), lr=0.001)

# Training loop
epochs = 30
for epoch in range(epochs):
    model_mlp.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model_mlp(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# Evaluation
model_mlp.eval()
correct = 0
total = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model_mlp(X_batch)
        _, predicted = torch.max(outputs.data, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

print(f"Accuracy of MLP on test data: {100 * correct / total:.2f}%")

Epoch 1/30, Loss: 0.7209
Epoch 2/30, Loss: 0.6034
Epoch 3/30, Loss: 0.6487
Epoch 4/30, Loss: 0.5718
Epoch 5/30, Loss: 0.5502
Epoch 6/30, Loss: 0.5773
Epoch 7/30, Loss: 0.6505
Epoch 8/30, Loss: 0.6832
Epoch 9/30, Loss: 0.6689
Epoch 10/30, Loss: 0.6687
Epoch 11/30, Loss: 0.5618
Epoch 12/30, Loss: 0.6062
Epoch 13/30, Loss: 0.6016
Epoch 14/30, Loss: 0.6001
Epoch 15/30, Loss: 0.6720
Epoch 16/30, Loss: 0.5726
Epoch 17/30, Loss: 0.7298
Epoch 18/30, Loss: 0.7170
Epoch 19/30, Loss: 0.6579
Epoch 20/30, Loss: 0.5990
Epoch 21/30, Loss: 0.6088
Epoch 22/30, Loss: 0.5626
Epoch 23/30, Loss: 0.6144
Epoch 24/30, Loss: 0.6193
Epoch 25/30, Loss: 0.5985
Epoch 26/30, Loss: 0.7025
Epoch 27/30, Loss: 0.5863
Epoch 28/30, Loss: 0.5774
Epoch 29/30, Loss: 0.5996
Epoch 30/30, Loss: 0.6290
Accuracy of MLP on test data: 64.87%


In [8]:
from pytorch_tabnet.tab_model import TabNetClassifier

# You need to convert data to numpy arrays for this library
X_train_np = X_train.values
y_train_np = y_train.values

X_test_np = X_test.values
y_test_np = y_test.values

# Initialize and train TabNet
# Note: You can tune various parameters like n_steps, n_d, n_a, etc.
clf = TabNetClassifier(
    n_steps=3,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size": 50, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='sparsemax', # or 'entmax'
    verbose=0
)

clf.fit(
    X_train=X_train_np,
    y_train=y_train_np,
    eval_set=[(X_test_np, y_test_np)],
    eval_metric=['accuracy'],
    max_epochs=200,
    patience=15,
)

# Make predictions
preds = clf.predict(X_test_np)
accuracy = (preds == y_test_np).sum() / len(y_test_np)
print(f"Accuracy of TabNet on test data: {accuracy:.4f}")


Early stopping occurred at epoch 45 with best_epoch = 30 and best_val_0_accuracy = 0.65096


/home/souparno/Documents/tennis-prediction/tennis-env/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Accuracy of TabNet on test data: 0.6510
